# E20 -- `telecomts_gap` end-to-end demo

Self-contained demo of the public audit toolkit released alongside the CIKM 2026 paper.

Runs four checks against synthetic Gaussian fixtures, so no external dataset download is needed:

1. `origin_audit` on two pools with the **same** distribution -> verdict `pass`.
2. `origin_audit` on two pools that occupy **different** operating regimes -> verdict `gap_detected`.
3. `origin_audit` on a benchmark with only **synthetic** anomalies (the operator-facing case from the paper's Section 4.3) -> verdict `origin_incomplete_synthetic_only`.
4. `calibration_budget` showing controlled-real recall recovery as the calibration fraction $f$ grows.
5. CLI + FastAPI invocations using the same machinery.

Total runtime on a 2024 laptop: ~30 s, no GPU required.

Prerequisite: `pip install -e .[api,dev]` from the repo root.

In [ ]:
import json
import numpy as np
import pandas as pd

from telecomts_gap import (
    Verdict,
    calibration_budget,
    origin_audit,
)

print("telecomts_gap import OK, verdict values:", [v.value for v in Verdict])

## 1. `origin_audit` -- two pools, same distribution

When the controlled-real and synthetic pools occupy the same KPI regime, the C2ST classifier cannot beat chance and MMD is small relative to the permutation null. The audit returns `Verdict.PASS`.

In [ ]:
def make_df(n_real: int, n_synth: int, n_feat: int, gap: float, seed: int = 0) -> pd.DataFrame:
    rng = np.random.default_rng(seed)
    X_real = rng.normal(loc=gap, scale=1.0, size=(n_real, n_feat))
    X_syn = rng.normal(loc=0.0, scale=1.0, size=(n_synth, n_feat))
    df = pd.DataFrame(
        np.vstack([X_real, X_syn]), columns=[f"kpi_{j}" for j in range(n_feat)]
    )
    df["anomaly_origin"] = ["controlled_real"] * n_real + ["synthetic"] * n_synth
    return df


df_pass = make_df(n_real=200, n_synth=200, n_feat=10, gap=0.0)
res_pass = origin_audit(df_pass, origin_col="anomaly_origin", n_perm=120)
print(json.dumps(res_pass.to_dict(), indent=2))

## 2. `origin_audit` -- two pools, different operating regimes

Shifting the controlled-real mean by 3 standard deviations puts the two pools in disjoint KPI regions. C2ST accuracy is near 1, MMD is many times the null mean, and the verdict flips to `gap_detected`.

In [ ]:
df_gap = make_df(n_real=200, n_synth=200, n_feat=10, gap=3.0)
res_gap = origin_audit(df_gap, origin_col="anomaly_origin", n_perm=120)
print(json.dumps(res_gap.to_dict(), indent=2))

## 3. `origin_audit` -- synthetic-only benchmark (operator-facing case)

The Section 4.3 verdict on an industrial timing-flow benchmark. The benchmark only ships synthetic anomalies; the audit refuses to certify it and emits `origin_incomplete_synthetic_only`.

In [ ]:
df_synth_only = make_df(n_real=0, n_synth=300, n_feat=10, gap=0.0)
res_incomplete = origin_audit(df_synth_only, origin_col="anomaly_origin")
print(json.dumps(res_incomplete.to_dict(), indent=2))
assert res_incomplete.verdict is Verdict.ORIGIN_INCOMPLETE_SYNTHETIC_ONLY

## 4. `calibration_budget` -- recall recovery curve

We construct a fixture where controlled-real and synthetic anomalies shift DIFFERENT features (mirroring the RSRP-regime mismatch identified in the paper). Synthetic-only training cannot transfer until at least a few controlled-real windows are added. The sweep below shows how controlled-real recall climbs with the calibration fraction $f$.

In [ ]:
rng = np.random.default_rng(0)
n_feat = 10

train_norm = rng.normal(0, 1, (600, n_feat))
train_synth = rng.normal(0, 1, (200, n_feat))
train_synth[:, 0] += 4.0
real_pool = rng.normal(0, 1, (60, n_feat))
real_pool[:, 1] += 4.0
test_norm = rng.normal(0, 1, (200, n_feat))
test_real = rng.normal(0, 1, (30, n_feat))
test_real[:, 1] += 4.0
test_synth = rng.normal(0, 1, (30, n_feat))
test_synth[:, 0] += 4.0

res = calibration_budget(
    train_norm,
    train_synth,
    real_pool,
    test_norm,
    test_real,
    test_synth,
    target_recall=0.7,
    sweep=(0.0, 0.05, 0.10, 0.25, 0.50, 1.00),
    n_seeds=5,
)

rows = [
    {
        "fraction": p.fraction,
        "n_added_real": p.n_added_real,
        "real_recall_mean": round(p.real_recall_mean, 3),
        "synth_recall_mean": round(p.synth_recall_mean, 3),
        "normal_fpr_mean": round(p.normal_fpr_mean, 3),
    }
    for p in res.sweep
]
print(pd.DataFrame(rows).to_string(index=False))
print()
print(f"Recommended fraction f for target_recall=0.7: {res.recommended_fraction}")
print(f"That corresponds to {res.recommended_n_added} controlled-real windows added to training.")

## 5. CLI and FastAPI -- the same machinery from two more surfaces

The CLI and FastAPI router call the same `origin_audit` function under the hood; they exist so CI pipelines and HTTP services can invoke the audit without depending on Python imports.

In [ ]:
import tempfile
from pathlib import Path

from telecomts_gap.cli import main as cli_main

tmp = Path(tempfile.mkdtemp())
df_gap.to_csv(tmp / "benchmark.csv", index=False)
out = tmp / "verdict.json"
rc = cli_main(
    [
        "--csv",
        str(tmp / "benchmark.csv"),
        "--output",
        str(out),
        "--n-perm",
        "80",
    ]
)
print(f"CLI exit code (0=pass, 1=gate fires): {rc}")
print("verdict JSON:", json.loads(out.read_text())["result"]["verdict"])

In [ ]:
try:
    from fastapi.testclient import TestClient

    from telecomts_gap.api import app

    client = TestClient(app)
    print("GET /audit/health ->", client.get("/audit/health").json())

    df_gap.to_csv(tmp / "benchmark.csv", index=False)
    with open(tmp / "benchmark.csv", "rb") as f:
        r = client.post(
            "/audit/origin",
            files={"file": ("benchmark.csv", f, "text/csv")},
            data={"synthetic_only_from_flag": "false", "n_perm": "80"},
        )
    print("POST /audit/origin ->", r.json()["result"]["verdict"])
except ImportError:
    print("FastAPI extra not installed -- run: pip install -e .[api]")

## What this demo shows

- The four verdict strings the paper quotes (`pass`, `gap_detected`, `origin_incomplete_synthetic_only`, `no_anomalies_present`) are real, machine-emitted strings -- no PDF-only text.
- `origin_audit` and `calibration_budget` are the same machinery that produced the Section 4.3 evidence files under [`evidence/industrial/`](../../evidence/industrial/).
- The CLI and FastAPI surfaces expose the same machinery so the audit can run as a CI gate or as a shadow-mode endpoint behind an existing service mesh.
- For the full reproducibility pipeline on TelecomTS / SpotLight, see the experiment table in the top-level [`README.md`](../../README.md).